# **Assignment 5: IKEA’s Digital Push: Optimizing Email Frequency and Personalization**

Wan Jun Wen, Vy (Valerie) Mai Le, Annie Tran, Lina Slaoui

BUS 390B: Marketing Analytics

In [ ]:
import statsmodels.formula.api as smf
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from patsy import build_design_matrices


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CHANGE THIS TO YOUR FOLDER
os.chdir('/content/drive/MyDrive/Ba390/Group Assignment 5')

print(os.getcwd())


Mounted at /content/drive
/content/drive/MyDrive/Ba390/Group Assignment 5


In [ ]:
df = pd.read_csv('IKEA.csv')
df


,customer_id,buy,order_size,message,age,female,income,home_improvement_index,space_optimization_index,number_of_Outdoor,number_of_HomeDecor,number_of_Bedroom,number_of_Storage,number_of_LivingRoom,number_of_Lighting,number_of_KitchenDining,training
0,6083030,0,0,Bedroom,>= 60,0,40000,24,0.4,0,0,0,1,0,0,1,1
1,7997633,1,108,Lighting,45 to 59,0,75000,50,1.0,0,0,2,3,3,0,1,1
2,1668615,0,0,LivingRoom,< 30,1,45000,42,0.4,0,0,1,2,2,4,2,1
3,1877853,1,442,Bedroom,45 to 59,0,75000,37,1.4,0,0,2,3,4,5,0,1
4,8425092,1,133,Outdoor,45 to 59,1,65000,38,1.4,0,0,1,2,3,1,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,9261480,0,0,LivingRoom,< 30,1,65000,34,1.0,0,0,1,0,3,4,6,0
249996,6553574,1,153,Bedroom,< 30,1,100000,77,1.0,0,0,4,8,10,9,9,0
249997,2591431,0,0,Lighting,>= 60,0,50000,24,1.1,0,0,0,1,0,1,0,0
249998,4694239,1,227,Lighting,30 to 44,1,70000,51,1.1,0,0,2,2,5,1,4,0


# **PART 1: Email Frequency**

**Understanding the patterns **


When IKEA sends more emails, customers think about the brand more often, which encourages them to look at products and make purchases. People who stay subscribed are usually the ones who already like IKEA and are more interested in buying, so their spending increases with more emails. Customers who unsubscribe may still buy things before they opt out, because earlier emails reminded them of products they need. Even after unsubscribing, some customers continue to shop at IKEA, and the ideas they saw in past emails can still influence their choices. These reasons explain why both groups spend more when email frequency increases.

**Compute six-month LTV**

In [ ]:
# Exhibit 1 data
email_data = pd.DataFrame({
    "emails_per_week": [1, 2, 3, 4],
    "unsub_rate": [0.024, 0.034, 0.097, 0.171],  # monthly unsubscribe rate
    "rev_sub": [9.97, 14.21, 14.56, 14.89],       # monthly revenue (subscribed)
    "rev_unsub": [6.32, 6.36, 6.70, 7.11]         # monthly revenue (unsubscribed)
})

# Constants
margin = 0.35              # 1 - 0.65 COGS
discount_rate = 0.01       # monthly discount rate

def compute_ltv(row):
    p_sub = 1.0   # start fully subscribed
    ltv = 0.0

    for t in range(1, 7):  # 6-month LTV
        expected_rev = p_sub * row.rev_sub + (1 - p_sub) * row.rev_unsub
        expected_profit = expected_rev * margin

        # discounting
        discounted_profit = expected_profit / ((1 + discount_rate) ** t)
        ltv += discounted_profit

        # update probability of remaining subscribed
        p_sub *= (1 - row.unsub_rate)

    return ltv

# Apply function
email_data["six_month_LTV"] = email_data.apply(compute_ltv, axis=1)

print(email_data)

   emails_per_week  unsub_rate  rev_sub  rev_unsub  six_month_LTV
0                1       0.024     9.97       6.32      19.797969
1                2       0.034    14.21       6.36      27.544832
2                3       0.097    14.56       6.70      26.169453
3                4       0.171    14.89       7.11      24.866134


**Recommend a single email-frequency policy**

Based on the six-month LTV results, IKEA should send 2 emails per week to all online customers. This frequency creates the highest LTV ($27.54) compared to 1, 3, or 4 emails per week. While sending more emails increases short-term revenue, it also leads to higher unsubscribe rates. At 3 and 4 emails per week, the unsubscribe rate becomes too high, which lowers long-term value.

Sending only 1 email per week reduces the unsubscribe rate, but it also limits sales and produces a much lower LTV. Therefore, 2 emails per week offers the best balance between customer engagement and retention, giving IKEA the strongest long-run financial outcome.

# **PART 2: What Should Each Customer’s Email Focus On? From Randomness to Personalization**


# Questions 1–4

1. Compute predicted probability of purchase for each of the seven departments  
2. Identify which department gives the highest probability  
3. Extend prediction from probability to profit  
4. Compare distributions of probability-based vs profit-based recommendations  


In [ ]:
O = 0.0084   # true buyer rate
R = 0.50     # oversampled buyer rate
COGS = 0.65  # cost of goods sold

train = df[df["training"] == 1].reset_index(drop=True)
valid = df[df["training"] == 0].reset_index(drop=True)

formula = (
    "buy ~ age + female + income + home_improvement_index + space_optimization_index "
    "+ number_of_LivingRoom + number_of_Bedroom + number_of_KitchenDining "
    "+ number_of_Storage + number_of_Lighting + number_of_HomeDecor + number_of_Outdoor"
)


# MODEL ESTIMATION FOR EACH DEPARTMENT
Below we estimate 7 weighted GLM models, one per department.


In [ ]:
# Bedroom
subset_B = train[train["message"]=="Bedroom"]
w1 = np.where(subset_B["buy"] == 1, O/R, (1-O)/(1-R))
model_B = smf.glm(formula, data=subset_B, family=sm.families.Binomial(), freq_weights=w1).fit()
valid["p_Bedroom"] = model_B.predict(valid)

# Home Decor
subset_H = train[train["message"]=="HomeDecor"]
w2 = np.where(subset_H["buy"] == 1, O/R, (1-O)/(1-R))
model_H = smf.glm(formula, data=subset_H, family=sm.families.Binomial(), freq_weights=w2).fit()
valid["p_HomeDecor"] = model_H.predict(valid)

# Kitchen Dining
subset_K = train[train["message"]=="KitchenDining"]
w3 = np.where(subset_K["buy"] == 1, O/R, (1-O)/(1-R))
model_K = smf.glm(formula, data=subset_K, family=sm.families.Binomial(), freq_weights=w3).fit()
valid["p_KitchenDining"] = model_K.predict(valid)

# Lighting
subset_L = train[train["message"]=="Lighting"]
w4 = np.where(subset_L["buy"] == 1, O/R, (1-O)/(1-R))
model_L = smf.glm(formula, data=subset_L, family=sm.families.Binomial(), freq_weights=w4).fit()
valid["p_Lighting"] = model_L.predict(valid)

# Living Room
subset_LR = train[train["message"]=="LivingRoom"]
w5 = np.where(subset_LR["buy"] == 1, O/R, (1-O)/(1-R))
model_LR = smf.glm(formula, data=subset_LR, family=sm.families.Binomial(), freq_weights=w5).fit()
valid["p_LivingRoom"] = model_LR.predict(valid)

# Outdoor
subset_O = train[train["message"]=="Outdoor"]
w6 = np.where(subset_O["buy"] == 1, O/R, (1-O)/(1-R))
model_O = smf.glm(formula, data=subset_O, family=sm.families.Binomial(), freq_weights=w6).fit()
valid["p_Outdoor"] = model_O.predict(valid)

# Storage
subset_S = train[train["message"]=="Storage"]
w7 = np.where(subset_S["buy"] == 1, O/R, (1-O)/(1-R))
model_S = smf.glm(formula, data=subset_S, family=sm.families.Binomial(), freq_weights=w7).fit()
valid["p_Storage"] = model_S.predict(valid)


In [ ]:
valid["p_max"] = valid[[
    "p_Bedroom", "p_HomeDecor", "p_KitchenDining",
    "p_Lighting", "p_LivingRoom", "p_Outdoor", "p_Storage"
]].max(axis=1)

valid["predicted_dep"] = valid[[
    "p_Bedroom", "p_HomeDecor", "p_KitchenDining",
    "p_Lighting", "p_LivingRoom", "p_Outdoor", "p_Storage"
]].idxmax(axis=1)

valid["predicted_dep"] = valid["predicted_dep"].str.replace("p_", "")
valid.head()


,customer_id,buy,order_size,message,age,female,income,home_improvement_index,space_optimization_index,number_of_Outdoor,...,training,p_Bedroom,p_HomeDecor,p_KitchenDining,p_Lighting,p_LivingRoom,p_Outdoor,p_Storage,p_max,predicted_dep
0,9342294,1,132,Outdoor,45 to 59,1,125000,72,1.2,0,...,0,0.160210,0.125041,0.169183,0.133938,0.256778,0.175846,0.149325,0.256778,LivingRoom
1,1253022,0,0,Outdoor,< 30,1,60000,42,1.1,0,...,0,0.008210,0.008454,0.011762,0.009231,0.010867,0.009051,0.007597,0.011762,KitchenDining
2,9970798,1,85,KitchenDining,>= 60,1,105000,73,1.3,0,...,0,0.120112,0.204982,0.155191,0.156778,0.176830,0.211041,0.129825,0.211041,Outdoor
3,7384371,1,66,Lighting,45 to 59,0,60000,34,1.2,0,...,0,0.019394,0.016617,0.017887,0.018345,0.018781,0.016508,0.016069,0.019394,Bedroom
4,7381423,0,0,KitchenDining,>= 60,0,55000,45,0.8,0,...,0,0.005900,0.005475,0.005723,0.004357,0.006673,0.005292,0.005288,0.006673,LivingRoom


# Distribution of departments chosen under probability-maximizing rule


In [ ]:
best_dep_dist = valid["predicted_dep"].value_counts(normalize=True) * 100
best_dep_dist


,proportion
predicted_dep,
LivingRoom,39.206
KitchenDining,37.198
Bedroom,9.736
Lighting,6.756
HomeDecor,4.458
Outdoor,2.202
Storage,0.444


# Profit-Based Recommendations
We now compute expected profit = probability × margin × average order size.
Margin = 1 - COGS = 35%.


In [ ]:
margin = 1 - COGS

avg_order = train.groupby("message")["order_size"].mean()

valid["prof_Bedroom"] = valid["p_Bedroom"] * margin * avg_order["Bedroom"]
valid["prof_HomeDecor"] = valid["p_HomeDecor"] * margin * avg_order["HomeDecor"]
valid["prof_KitchenDining"] = valid["p_KitchenDining"] * margin * avg_order["KitchenDining"]
valid["prof_Lighting"] = valid["p_Lighting"] * margin * avg_order["Lighting"]
valid["prof_LivingRoom"] = valid["p_LivingRoom"] * margin * avg_order["LivingRoom"]
valid["prof_Outdoor"] = valid["p_Outdoor"] * margin * avg_order["Outdoor"]
valid["prof_Storage"] = valid["p_Storage"] * margin * avg_order["Storage"]

profit_cols = [
    "prof_Bedroom", "prof_HomeDecor", "prof_KitchenDining",
    "prof_Lighting", "prof_LivingRoom", "prof_Outdoor",
    "prof_Storage"
]

valid["best_profit_department"] = valid[profit_cols].idxmax(axis=1).str.replace("prof_", "")
valid.head()


,customer_id,buy,order_size,message,age,female,income,home_improvement_index,space_optimization_index,number_of_Outdoor,...,p_max,predicted_dep,prof_Bedroom,prof_HomeDecor,prof_KitchenDining,prof_Lighting,prof_LivingRoom,prof_Outdoor,prof_Storage,best_profit_department
0,9342294,1,132,Outdoor,45 to 59,1,125000,72,1.2,0,...,0.256778,LivingRoom,5.794213,4.504213,6.058799,4.686093,8.577449,6.248613,5.341566,LivingRoom
1,1253022,0,0,Outdoor,< 30,1,60000,42,1.1,0,...,0.011762,KitchenDining,0.296926,0.304532,0.421229,0.322978,0.363009,0.321634,0.271750,KitchenDining
2,9970798,1,85,KitchenDining,>= 60,1,105000,73,1.3,0,...,0.211041,Outdoor,4.344034,7.383851,5.557730,5.485201,5.906846,7.499236,4.644034,Outdoor
3,7384371,1,66,Lighting,45 to 59,0,60000,34,1.2,0,...,0.019394,Bedroom,0.701428,0.598589,0.640556,0.641831,0.627360,0.586587,0.574816,Bedroom
4,7381423,0,0,KitchenDining,>= 60,0,55000,45,0.8,0,...,0.006673,LivingRoom,0.213375,0.197204,0.204955,0.152455,0.222891,0.188041,0.189174,LivingRoom


# Distribution of profit-maximizing recommendations


In [ ]:
best_dep_dist2 = valid["best_profit_department"].value_counts(normalize=True) * 100
best_dep_dist2


,proportion
best_profit_department,
KitchenDining,43.778
LivingRoom,24.502
Bedroom,14.680
HomeDecor,7.110
Lighting,6.576
Outdoor,2.564
Storage,0.790


Differences between probability vs profit distributions:
- KitchenDining becomes the best department under profit.
- LivingRoom falls significantly in the profit-based method.
- HomeDecor increases.
- Outdoor and Storage stay low in both.


# Questions 5–8:

In this section, we extend the analysis from predicted probabilities and profit per department to:

5. Compute the average expected profit per customer under a fully personalized strategy.  
6. Compute the average expected profit per customer if IKEA sends the same department to everyone (one-size-fits-all).  
7. Compute the average expected profit per customer if departments are assigned at random with equal probability.  
8. Compare personalized vs. random assignment and estimate the percentage and total dollar improvement for a national campaign of 10 million customers.



## Question 5 — Average Expected Profit Under Personalization

For each customer, we already have seven expected profit values:

- `prof_Bedroom`, `prof_HomeDecor`, `prof_KitchenDining`,  
- `prof_Lighting`, `prof_LivingRoom`, `prof_Outdoor`, `prof_Storage`.

Each one is:  

\[
\text{Expected Profit}_{i,d} = P_{i,d} \times \text{margin} \times \text{avg order size}_d
\]

To obtain the personalized strategy, we assign each customer the **department that gives the highest expected profit** and then average across customers.


Q5 computation

In [ ]:
# Expected profit using best department per customer (personalized strategy)
valid["personalized_profit"] = valid[profit_cols].max(axis=1)

# Average expected profit per customer under personalization
avg_personalized_profit = valid["personalized_profit"].mean()
avg_personalized_profit


np.float64(2.044176417025179)

Each customer generates about $2.044 in expected profit.

## Question 6 — Average Expected Profit Under One-Size-Fits-All Strategy

In this scenario, IKEA chooses *one single department* and sends that email to **every** customer.
For each department \( d \), the expected profit per customer is:

\[
\text{Avg Profit}_d = \frac{1}{N} \sum_{i=1}^{N} \text{Expected Profit}_{i,d}
\]

We compute the mean of each `prof_` column.  
This allows us to identify:

- the most profitable department if IKEA uses a one-size-fits-all message,
- the relative profitability of each department.


In [ ]:
# Average expected profit per customer if we always send the same department
avg_profit_by_dept = {col.replace("prof_", ""): valid[col].mean() for col in profit_cols}
avg_profit_by_dept


{'Bedroom': np.float64(1.5456687360321841),
 'HomeDecor': np.float64(1.4962662132284104),
 'KitchenDining': np.float64(1.9103007259663645),
 'Lighting': np.float64(1.6283538929784502),
 'LivingRoom': np.float64(1.6526521455220462),
 'Outdoor': np.float64(1.7622592777984392),
 'Storage': np.float64(1.6422386592002853)}

In [ ]:
best_uniform_department = max(avg_profit_by_dept, key=avg_profit_by_dept.get)
best_uniform_profit = avg_profit_by_dept[best_uniform_department]
best_uniform_department, best_uniform_profit


('KitchenDining', np.float64(1.9103007259663645))

### Question 6 — One-Size-Fits-All Expected Profit

To evaluate the performance of a uniform strategy, we computed the average expected profit per
customer for each department if IKEA were to send the same message to everyone.

The results show that **KitchenDining** produces the highest expected profit, approximately
**$1.914 per customer**, outperforming all other departments such as LivingRoom, Lighting, Outdoor, etc.

This suggests that if IKEA were limited to using a single message for all customers, sending a
KitchenDining promotion would yield the greatest profitability.


Run Question 7

In [ ]:
valid["random_profit"] = valid[profit_cols].mean(axis=1)
avg_random_profit = valid["random_profit"].mean()
avg_random_profit


np.float64(1.6625342358180255)

### Question 7 — Expected Profit Under Random Assignment

In the random assignment scenario, each customer is assigned one of the seven departments with
equal probability (1/7). For each customer, we took the average of the seven expected profit
values and then averaged this quantity across all customers.

The resulting expected profit under random messaging is approximately **$1.663 per customer**.

This value serves as the baseline benchmark to compare the performance of both the personalized
strategy (from Question 5) and the best uniform strategy (from Question 6).


Question 8

In [ ]:
# Percentage improvement of personalized vs random
improvement_pct = (avg_personalized_profit - avg_random_profit) / avg_random_profit * 100

# Dollar improvement for a national campaign of 10 million customers
total_customers = 10_000_000
total_lift = (avg_personalized_profit - avg_random_profit) * total_customers

improvement_pct, total_lift


(np.float64(22.955447953188894), np.float64(3816421.8120715353))

### Question 8 — Improvement from Personalization vs. Random Assignment

To quantify the benefit of personalization, we compare the expected profit from the personalized
strategy (Question 5) to the expected profit generated under random assignment (Question 7).

The personalized strategy yields an average expected profit of approximately **$2.044 per
customer**, compared to **$1.663** under random assignment. This corresponds to an improvement
of about **22.96%**:

\[
\text{Improvement} =
\frac{2.044 - 1.663}{1.663} \times 100 \approx 22.96\%
\]

To assess the practical impact, we scale this increase to a national email campaign targeting  
**10 million customers**:

\[
\text{Total Lift} = (2.044 - 1.663) \times 10{,}000{,}000
\approx \$3{,}816{,}422
\]

Thus, adopting a personalized messaging strategy could generate **about \$3.82 million in
additional profit**, compared to assigning departments at random.
